# Semantic Segmentation: U-Net Architecture, Dice Loss & mIoU Evaluation

This interactive lab covers:
1. **Encoder-Decoder Networks**: U-Net architecture with high-resolution skip connections.
2. **Transposed Convolutions**: Learnable spatial upsampling in expanding decoder paths.
3. **Loss Functions for Spatial Imbalance**: Cross-Entropy vs. Soft Dice Loss.
4. **Evaluation Metrics**: Per-class Intersection-over-Union and Mean IoU (mIoU).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('04-computer-vision/semantic-segmentation/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.nn.functional as F
from segmentation_engine import (
    UNetMini,
    DiceLoss,
    compute_mean_iou,
)

print('Semantic Segmentation engine loaded successfully!')

## 1. U-Net Architecture Inspection

We instantiate a lightweight U-Net with contracting encoder stages, bottleneck, and expanding decoder stages with skip connections.

In [ ]:
torch.manual_seed(42)
num_classes = 2  # Background vs. Foreground Object
unet = UNetMini(in_channels=3, num_classes=num_classes, base_features=16)

# Input image batch: (B=2, C=3, H=32, W=32)
images = torch.randn(2, 3, 32, 32)
logits = unet(images)

print(f'Input image batch shape:   {images.shape}')
print(f'Output mask logits shape:  {logits.shape}')
assert logits.shape == (2, num_classes, 32, 32)

## 2. Loss Functions: Cross-Entropy vs. Soft Dice Loss

In medical and satellite imagery, target objects occupy only a tiny fraction of pixels. Soft Dice Loss directly optimizes overlap.

In [ ]:
# Synthetic highly imbalanced targets: 95% background (0), 5% target object (1)
targets = torch.zeros(2, 32, 32, dtype=torch.long)
targets[:, 14:18, 14:18] = 1  # 16 foreground pixels

ce_loss = F.cross_entropy(logits, targets)
dice_loss_fn = DiceLoss()
d_loss = dice_loss_fn(logits, targets)

print(f'Cross-Entropy Loss: {ce_loss.item():.4f}')
print(f'Soft Dice Loss:     {d_loss.item():.4f}')

## 3. Backpropagation & Optimization Step

Verifying end-to-end gradient flow through decoder, skip connections, and encoder stem.

In [ ]:
optimizer = torch.optim.Adam(unet.parameters(), lr=1e-3)
optimizer.zero_grad()
combined_loss = ce_loss + d_loss
combined_loss.backward()
optimizer.step()

stem_grad_norm = unet.inc.conv[0].weight.grad.norm().item()
print(f'Encoder stem gradient norm: {stem_grad_norm:.6f}')
assert stem_grad_norm > 0, 'Gradients must flow to encoder input layer!'

## 4. Evaluation: Mean Intersection-over-Union (mIoU)

Computing per-class IoU and mean IoU across predicted binary masks.

In [ ]:
pred_masks = torch.argmax(logits.detach(), dim=1)
mean_iou, class_ious = compute_mean_iou(pred_masks, targets, num_classes=2)

print(f'Class 0 (Background) IoU: {class_ious[0].item():.4f}')
print(f'Class 1 (Foreground) IoU: {class_ious[1].item():.4f}')
print(f'Mean IoU (mIoU):          {mean_iou:.4f}')